# Dealing with Lookahead Conflicts

This notebook discusses conflicts that have their origin in insufficient lookahead.
We will discuss the following grammar:
```
    a : b "U" "V"
      | c "U" "W"

    b : "X"

    c : "X"
```
This grammar is *unambiguous*:  There are only two string that can be derived from the start symbol `a`:
 * `"XUV"`,
 * `"XUW"`.

Each of these strings has exactly one parse tree.
Nevertheless, the grammar does not have the `LR(1)` property and is therefore certainly not an `LALR(1)` grammar.

## Specification of the Grammar

Below is the grammar from above written in the format for `Lark`.

In [ ]:
grammar = r"""
    a : b "U" "V"
      | c "U" "W"

    b : "X"

    c : "X"
"""

In order to see what is going on when we try to build the parser, we have to activate logging.

In [ ]:
import logging

from lark import Lark, GrammarError, logger

logger.setLevel(logging.DEBUG)

## Trying to Build an `LALR` Parser

When we try to build a parser for a grammar that contains a *reduce/reduce* conflict, `Lark` raises an exception of class `GrammarError`.  This is independent of the keyword argument `strict`:  in contrast to a *shift/reduce* conflict, which is resolved in favour of shifting, there is no default resolution for a *reduce/reduce* conflict.  Instead, a 
*reduce/reduce* conflict always throws an exception.

In order to shows the offending state, we have to create the parser created with the option `debug=True`.

In [ ]:
try:
    Lark(grammar, start='a', parser='lalr', debug=True)
    print('No conflict.')
except GrammarError as e:
    print(e)

## Why Is There a Conflict?

If we compute the `LR` states of this grammar, we find, among others, the state
$$ \{\; b \rightarrow \texttt{'X'} \bullet : \texttt{'U'}, \quad
       c \rightarrow \texttt{'X'} \bullet : \texttt{'U'} \;\}, $$
which is exactly the state shown in the error message above.  Since the set of follow tokens is the same for
both rules, we have a *reduce/reduce* conflict here.

The conflict is caused by the fact that the parser, with a lookahead of only *one* token, cannot decide
whether an `'X'` should be interpreted as a `b` or as a `c`, because this is only decided when the character
*following* the `'U'` is read:

* if this is a `'V'`, then the rule $a \rightarrow b\,\texttt{'U'}\,\texttt{'V'}$ is used and consequently
  the `'X'` has to be interpreted as a `b`;
* if, however, the second token after the `'X'` is a `'W'`, then the rule
  $a \rightarrow c\,\texttt{'U'}\,\texttt{'W'}$ is used and consequently the `'X'` is to be read as a `c`.

Note that there is no parse table to inspect:  since the exception is raised *during* the construction of the
table, there is no parser object and therefore nothing that `dump_states` could be applied to.

## A Way Out: The `Earley` Parser

`Lark` offers an escape hatch that the parser generators of the `yacc` family do not have.  The *default*
parser of `Lark` is an `Earley` parser, and the `Earley` algorithm does not need a bounded lookahead at all.
Therefore, the grammar is accepted as it stands.

The keyword argument `keep_all_tokens=True` tells `Lark` to keep the anonymous terminals in the parse tree.
Without it, the tokens `'U'`, `'V'`, `'W'`, and `'X'` would be filtered out and the trees below would be
almost empty.

The price for this convenience is a parser that is considerably slower than an `LALR` parser.

In [ ]:
earley_parser = Lark(grammar, start='a', parser='earley', keep_all_tokens=True)

The `Earley` parser has no difficulty deciding whether the `'X'` is a `b` or a `c`:  it simply pursues both
possibilities until the last token settles the question.

In [ ]:
print(earley_parser.parse('XUV').pretty())

In [ ]:
print(earley_parser.parse('XUW').pretty())

## The Real Solution

Switching to an `Earley` parser hides the problem instead of solving it.  The grammar can be rewritten so
that it becomes an `LALR(1)` grammar:  if the context `'U'`, which follows both `b` and `c`, is pulled into
the rules for `b` and `c`, then the conflict disappears, because the state in which the conflict occurred now
has the form
$$ \{\; b \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'V'}, \quad
       c \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'W'} \;\} $$
and here the next token decides which rule to reduce with.  This is done in the notebook
[04-Look-Ahead-Solved.ipynb](04-Look-Ahead-Solved.ipynb).